In [ ]:
from data import generate_quad_gmm

from flower.evaluation.metrics import evaluate_embedding_classifier, evaluate_embedding_regressor
from flower.evaluation.benchmarks import resiualize_categorical 

from sklearn.linear_model import LinearRegression # LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
import umap
import numpy as np 

RANDOM_STATE = 42

In [3]:
X_train, y_train, yr_train, _ = generate_quad_gmm(20000)
X_test, y_test, yr_test, _ = generate_quad_gmm(5000)

In [4]:
clf_models = {
    '1-mlp': MLPClassifier(
        hidden_layer_sizes=(64,), # Logistic Regression
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    '2-mlp': MLPClassifier(
        hidden_layer_sizes=(64, 64),  # 32,16
        max_iter=1000, 
        random_state=RANDOM_STATE
    )
}

reg_models = {
    '1-mlp': MLPRegressor(
        hidden_layer_sizes=(64,),
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    '2-mlp': MLPRegressor(
        hidden_layer_sizes=(64, 64), 
        max_iter=1000, 
        random_state=42
    )
}

# Baseline

In [ ]:
for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )
    print("-----")

In [ ]:
for model_name, reg_model in reg_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_regressor(
        X_train=X_train,
        X_test=X_test,
        y_train=yr_train,
        y_test=yr_test,
        model=reg_model
    )

## Linear Residualisation

In [7]:
linear_model = LinearRegression()
X_train_res, X_test_res = resiualize_categorical(
    X_train, 
    y_train.reshape(-1, 1), 
    X_test, 
    y_test.reshape(-1, 1), 
    linear_model
)

In [ ]:
for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )

In [ ]:
for model_name, reg_model in reg_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_regressor(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=yr_train,
        y_test=yr_test,
        model=reg_model
    )

# MLP Resdiualisation

In [10]:
mlp_model = MLPRegressor(
    hidden_layer_sizes=(64, 64),
    max_iter=1000,
    random_state=42
)
X_train_res, X_test_res = resiualize_categorical(
    X_train, 
    y_train.reshape(-1, 1), 
    X_test, 
    y_test.reshape(-1, 1), 
    mlp_model
)

In [ ]:
for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )

In [ ]:
for model_name, reg_model in reg_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_regressor(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=yr_train,
        y_test=yr_test,
        model=reg_model
    )

## Random Forest Residualisation

In [13]:
rf_model = RandomForestRegressor(
    n_estimators=100
)
X_train_res, X_test_res = resiualize_categorical(
    X_train, 
    y_train.reshape(-1, 1), 
    X_test, 
    y_test.reshape(-1, 1), 
    rf_model
)

In [ ]:
for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )

In [ ]:
for model_name, reg_model in reg_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_regressor(
        X_train=X_train_res,
        X_test=X_test_res,
        y_train=yr_train,
        y_test=yr_test,
        model=reg_model
    )

## Conditional UMAP

In [ ]:
reducer = umap.UMAP(
    n_components=10, 
    n_neighbors=30, 
    random_state=RANDOM_STATE, 
    transform_seed=42, 
    verbose=False
)

reducer.fit(X_train, y_train)

# 3. Transform both train and validation sets
c_embed_train = reducer.transform(X_train)
c_embed_test = reducer.transform(X_test)

In [ ]:
for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=c_embed_train,
        X_test=c_embed_test,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )

In [ ]:
for model_name, reg_model in reg_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_regressor(
        X_train=c_embed_train,
        X_test=c_embed_test,
        y_train=yr_train,
        y_test=yr_test,
        model=reg_model
    )

# Flower's Conditional Representation

In [ ]:
import os

import torch
from model import ConditionedVelocityModelWrapper

from flow_matching.solver import ODESolver
import torch.distributions as D
from model import MLP, Prior

device = torch.device("cpu")
gaussian_log_density = D.Independent(
    D.Normal(torch.zeros(2, device=device), torch.ones(2, device=device)), 1
).log_prob

vf = MLP(dim=2, h=64).to(device)
prior = Prior()

# 1. Define the folder and filename
folder_path = './checkpoints/' # Change this to your desired folder
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

checkpoint_path = os.path.join(folder_path, 'cond_fm.pth')

ckpt = torch.load(checkpoint_path)
vf.load_state_dict(ckpt["vf_state_dict"])
prior.load_state_dict(ckpt["prior_state_dict"])

vf = vf.to(device)
prior = prior.to(device)

In [22]:
n_steps = 200
wrapped_vf = ConditionedVelocityModelWrapper(
    velocity_model=vf, 
    cfg_scale=1.0
)
solver = ODESolver(velocity_model=wrapped_vf)

# We only care about the result at the end of the 1 -> 0 path
X_flow_train = solver.compute_likelihood(
    x_1=X_train.to(device),
    y=y_train.to(device),
    time_grid=torch.linspace(1, 0, n_steps),
    method="midpoint",
    step_size=None,
    exact_divergence=False,
    log_p0=gaussian_log_density,
    return_intermediates=False,
)[0].detach().cpu().numpy()

X_flow_test = solver.compute_likelihood(
    x_1=X_test.to(device),
    y=y_test.to(device),
    time_grid=torch.linspace(1, 0, n_steps),
    method="midpoint",
    step_size=None,
    exact_divergence=False,
    log_p0=gaussian_log_density,
    return_intermediates=False,
)[0].detach().cpu().numpy()

In [ ]:
for model_name, clf_model in clf_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_classifier(
        X_train=X_flow_train,
        X_test=X_flow_test,
        y_train=y_train,
        y_test=y_test,
        model=clf_model
    )

In [ ]:
for model_name, reg_model in reg_models.items():
    print(f"Model name: {model_name}")
    evaluate_embedding_regressor(
        X_train=X_flow_train,
        X_test=X_flow_test,
        y_train=yr_train,
        y_test=yr_test,
        model=reg_model
    )